In [ ]:
# Import Data Science Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

# Tensorflow Libraries
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import Model

# System libraries
from pathlib import Path
import os.path

# Metrics
from sklearn.metrics import classification_report

# Reproducibility
tf.random.set_seed(42)

In [ ]:
from helper_functions import (
    create_tensorboard_callback,
    plot_loss_curves,
    unzip_data,
    compare_historys,
    walk_through_dir,
    pred_and_plot,
    make_confusion_matrix,
    calculate_results,
)

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

In [ ]:
dataset = "./fire_dataset"
walk_through_dir(dataset)

In [ ]:
image_dir = Path(dataset)


filepaths = list(image_dir.glob(r'**/*.JPG')) + list(image_dir.glob(r'**/*.jpg')) + list(image_dir.glob(r'**/*.png'))

labels = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepaths))

filepaths = pd.Series(filepaths, name='Filepath').astype(str)
labels = pd.Series(labels, name='Label')


image_df = pd.concat([filepaths, labels], axis=1)

In [ ]:
len(list(image_dir.glob(r'**/*.png')))

In [ ]:
image_df


In [ ]:
random_index = np.random.randint(0, len(image_df), 16)
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(10, 10),
                        subplot_kw={'xticks': [], 'yticks': []})

for i, ax in enumerate(axes.flat):
    image = Image.open(image_df.Filepath[random_index[i]])
    ax.imshow(image)
    ax.set_title(image_df.Label[random_index[i]])
plt.tight_layout()
plt.show()

In [ ]:
train_df, test_df = train_test_split(image_df, test_size=0.2, shuffle=True, random_state=42)


In [ ]:
train_generator = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2],
)
test_generator = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

In [ ]:
train_images = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMAGE_SIZE,
    color_mode='rgb',
    class_mode='binary',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    subset='training'
)
val_images = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMAGE_SIZE,
    color_mode='rgb',
    class_mode='binary',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    subset='validation'
)
test_images = test_generator.flow_from_dataframe(
    dataframe=test_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMAGE_SIZE,
    color_mode='rgb',
    class_mode='binary',
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
idx_to_class = dict((v, k) for k, v in train_images.class_indices.items())

y_train_numeric = np.array(
    [train_images.class_indices[label] for label in train_df.Label]
)
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train_numeric
)
class_weight = {0: weights[0], 1: weights[1]}

print(f'Mapping classes : {idx_to_class}')
print(f'Poids des classes : {class_weight}')

In [ ]:
pretrained_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

pretrained_model.trainable = False

In [ ]:
checkpoint_path = "mobilenet_v2.weights.h5"
checkpoint_callback = ModelCheckpoint(
    checkpoint_path,
    save_weights_only=True,
    monitor="val_loss",
    mode="min",
    save_best_only=True
)

In [ ]:
# EarlyStopping : arret si val_loss ne s'ameliore pas pendant 5 epochs
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3))
x = pretrained_model(inputs, training=False)

x = Dense(256, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_images,
    steps_per_epoch=len(train_images),
    validation_data=val_images,
    validation_steps=len(val_images),
    epochs=100,
    class_weight=class_weight,
    callbacks=[
        early_stopping,
        create_tensorboard_callback('training_logs', 'fire_classification'),
        checkpoint_callback,
    ]
)

In [ ]:
model.save('fire_model.keras')
print('Modele sauvegarde : fire_model.keras')

In [ ]:
results = model.evaluate(test_images, verbose=0)

print("    Test Loss: {:.5f}".format(results[0]))
print("Test Accuracy: {:.2f}%".format(results[1] * 100))

In [ ]:
plot_loss_curves(history)

In [ ]:
# Phase 2 : fine-tuning — degeler les 30 derniers layers de MobileNetV2
pretrained_model.trainable = True
for layer in pretrained_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_images,
    steps_per_epoch=len(train_images),
    validation_data=val_images,
    validation_steps=len(val_images),
    epochs=50,
    class_weight=class_weight,
    callbacks=[
        early_stopping,
        create_tensorboard_callback('training_logs', 'fire_fine_tuning'),
        checkpoint_callback,
    ]
)

compare_historys(
    history,
    history_fine,
    initial_epochs=len(history.history['loss'])
)

In [ ]:
from sklearn.metrics import roc_curve, auc as sklearn_auc

pred_proba = model.predict(test_images).flatten()
y_true_binary = np.array([train_images.class_indices[label] for label in test_df.Label])

# Classe positive = fire_images
fire_class_idx = train_images.class_indices['fire_images']
fire_score = pred_proba if fire_class_idx == 1 else 1 - pred_proba
y_fire = (y_true_binary == fire_class_idx).astype(int)

# Courbe ROC et seuil optimal (critère de Youden : max TPR - FPR)
fpr, tpr, thresholds_roc = roc_curve(y_fire, fire_score)
roc_auc = sklearn_auc(fpr, tpr)
optimal_idx = np.argmax(tpr - fpr)

# Reconvertir en seuil sur pred_proba
optimal_threshold = thresholds_roc[optimal_idx] if fire_class_idx == 1 else 1 - thresholds_roc[optimal_idx]

print(f"AUC-ROC                : {roc_auc:.4f}")
print(f"Seuil par défaut       : 0.5000")
print(f"Seuil optimal (Youden) : {optimal_threshold:.4f}")

pred_binary = (pred_proba >= optimal_threshold).astype(int).flatten()
pred = [idx_to_class[k] for k in pred_binary]

print(f'Les 5 premières prédictions : {pred[:5]}')

In [ ]:
# Courbe ROC — visualisation
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, ls='--', label='Classifieur aléatoire')
plt.scatter(fpr[optimal_idx], tpr[optimal_idx], marker='*', s=250, color='red', zorder=5,
            label=f'Seuil optimal = {optimal_threshold:.4f}')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de Faux Positifs (FPR)')
plt.ylabel('Taux de Vrais Positifs / Recall (TPR)')
plt.title("Courbe ROC — Détection d'incendie")
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
random_index = np.random.randint(0, len(test_df), 15)
fig, axes = plt.subplots(nrows=3, ncols=5, figsize=(25, 15),
                        subplot_kw={'xticks': [], 'yticks': []})

for i, ax in enumerate(axes.flat):
    image = Image.open(test_df.Filepath.iloc[random_index[i]])
    ax.imshow(image)
    if test_df.Label.iloc[random_index[i]] == pred[random_index[i]]:
      color = "green"
    else:
      color = "red"
    ax.set_title(f"True: {test_df.Label.iloc[random_index[i]]}\nPredicted: {pred[random_index[i]]}", color=color)
plt.show()
plt.tight_layout()

In [ ]:
y_test = list(test_df.Label)
print(classification_report(y_test, pred))

In [ ]:
report = classification_report(y_test, pred, output_dict=True)
df = pd.DataFrame(report).transpose()
df

In [ ]:
results_dict = calculate_results(y_test, pred)
print('Metriques detaillees :')
for metric, value in results_dict.items():
    print(f'  {metric}: {value:.4f}')

In [ ]:
make_confusion_matrix(
    y_test,
    pred,
    classes=list(train_images.class_indices.keys()),
    figsize=(8, 6),
    text_size=12
)

In [ ]:
# Inference sur une image unique — remplacer par n'importe quel chemin
sample_image_path = test_df.Filepath.iloc[0]
class_names = list(train_images.class_indices.keys())
pred_and_plot(model, sample_image_path, class_names)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image


def make_gradcam_heatmap(img_preprocessed, model, pretrained_model, last_conv_layer_name="out_relu"):
    """
    Grad-CAM adapte a l'architecture MobileNetV2 + Dense head + sortie sigmoid binaire.

    Cle : on applique un GAP manuel sur conv_maps pour garder le chemin de gradient
    continu depuis les feature maps jusqu'a la prediction.
    """
    # Sous-modele : entree → feature maps spatiales (None, 7, 7, 1280)
    conv_model = tf.keras.Model(
        inputs=pretrained_model.inputs,
        outputs=pretrained_model.get_layer(last_conv_layer_name).output,
    )

    # Couches de la tete du modele principal (apres MobileNetV2)
    # model.layers : [InputLayer, MobileNetV2, Dense(256), Dropout, Dense(1)]
    head_layers = model.layers[2:]

    img_tensor = tf.cast(img_preprocessed, tf.float32)

    with tf.GradientTape() as tape:
        conv_maps = conv_model(img_tensor, training=False)
        tape.watch(conv_maps)

        # GAP manuel — maintient le gradient path a travers conv_maps
        x = tf.reduce_mean(conv_maps, axis=[1, 2])

        # Passer dans la tete Dense
        for layer in head_layers:
            x = layer(x, training=False)

        loss = x[:, 0]  # scalaire sigmoide

    # Gradient de la prediction par rapport aux feature maps
    grads = tape.gradient(loss, conv_maps)  # (1, 7, 7, 1280)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))  # (1280,)

    # Combinaison lineaire ponderee des feature maps
    heatmap = conv_maps[0] @ pooled_grads[..., tf.newaxis]  # (7, 7, 1)
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()


def show_gradcam(image_path, model, pretrained_model, class_names, alpha=0.45):
    """Affiche image originale | heatmap | superposition Grad-CAM."""

    # Preprocessing
    img_pil = Image.open(image_path).resize((224, 224)).convert("RGB")
    img_array = np.array(img_pil, dtype=np.float32)
    img_preprocessed = np.expand_dims(
        tf.keras.applications.mobilenet_v2.preprocess_input(img_array.copy()), axis=0
    )

    # Prediction
    pred_proba = float(model.predict(img_preprocessed, verbose=0)[0][0])
    pred_idx = int(pred_proba > 0.5)
    pred_class = class_names[pred_idx]
    confidence = pred_proba if pred_idx == 1 else 1.0 - pred_proba

    # Heatmap
    heatmap = make_gradcam_heatmap(img_preprocessed, model, pretrained_model)

    # Superposition
    heatmap_up = np.array(Image.fromarray(np.uint8(255 * heatmap)).resize((224, 224)))
    heatmap_rgb = plt.cm.jet(heatmap_up / 255.0)[:, :, :3]
    superimposed = np.clip(alpha * heatmap_rgb + (1 - alpha) * img_array / 255.0, 0, 1)

    # Affichage
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_array / 255.0);   axes[0].set_title("Image originale");   axes[0].axis("off")
    axes[1].imshow(heatmap, cmap="jet"); axes[1].set_title("Heatmap Grad-CAM");  axes[1].axis("off")
    axes[2].imshow(superimposed)
    axes[2].set_title(f"Superposition\nPrediction : {pred_class} ({confidence:.1%})")
    axes[2].axis("off")

    plt.suptitle("Grad-CAM — zones d'attention du modele", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


# ── Utilisation ───────────────────────────────────────────────────────────────
show_gradcam(
    image_path=test_df.Filepath.iloc[0],
    model=model,
    pretrained_model=pretrained_model,
    class_names=list(train_images.class_indices.keys()),
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf

random_indices = np.random.choice(len(test_df), 9, replace=False)
class_names = list(train_images.class_indices.keys())

fig, axes = plt.subplots(3, 3, figsize=(14, 14))

for ax, idx in zip(axes.flat, random_indices):
    path = test_df.Filepath.iloc[idx]
    true_label = test_df.Label.iloc[idx]

    # Preprocessing
    img_pil = Image.open(path).resize((224, 224)).convert("RGB")
    img_array = np.array(img_pil, dtype=np.float32)
    img_preprocessed = np.expand_dims(
        tf.keras.applications.mobilenet_v2.preprocess_input(img_array.copy()), axis=0
    )

    # Prediction
    proba = float(model.predict(img_preprocessed, verbose=0)[0][0])
    pred_label = class_names[int(proba > 0.5)]
    confidence = proba if proba > 0.5 else 1.0 - proba

    # Couleur selon bonne/mauvaise prediction
    correct = pred_label == true_label
    color = "green" if correct else "red"
    icon = "FEU" if "fire_images" == pred_label else "PAS DE FEU"

    ax.imshow(img_array / 255.0)
    ax.set_title(
        f"Prediction : {icon}\nConfiance : {confidence:.1%}\nVrai label : {true_label}",
        color=color,
        fontsize=11,
        fontweight="bold",
    )
    ax.axis("off")

plt.suptitle("9 images aleatoires — vert = correct, rouge = erreur", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Correction de la ligne tronquée
gap_loss = best_val_loss - final_train_loss

test_results = model.evaluate(test_images, verbose=0)
test_loss, test_acc = test_results[0], test_results[1]

print(f"  Train loss      : {final_train_loss:.4f}")
print(f"  Val   loss      : {best_val_loss:.4f}")
print(f"  Écart loss      : {gap_loss:+.4f}   {' overfitting probable' if gap_loss > 0.1 else ' OK'}")
print(f"\nTest accuracy   : {test_acc:.4f}")
print(f"Test loss       : {test_loss:.4f}")
print(f"Écart val/test accuracy : {best_val_acc - test_acc:+.4f}   "
      f"{' val set trop optimiste' if best_val_acc - test_acc > 0.05 else ' OK'}")

# ── Graphiques ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

epochs_ran = len(history.history["accuracy"])
train_acc  = history.history["accuracy"]
val_acc    = history.history["val_accuracy"]
train_loss = history.history["loss"]
val_loss   = history.history["val_loss"]
epochs     = range(1, epochs_ran + 1)

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, train_acc, label="Train", linewidth=2)
ax1.plot(epochs, val_acc,   label="Validation", linewidth=2, linestyle="--")
ax1.axvline(best_epoch + 1, color="gray", linestyle=":", label=f"Meilleure epoch ({best_epoch+1})")
ax1.set_title("Accuracy — Train vs Validation"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.legend(); ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs, train_loss, label="Train", linewidth=2)
ax2.plot(epochs, val_loss,   label="Validation", linewidth=2, linestyle="--")
ax2.axvline(best_epoch + 1, color="gray", linestyle=":", label=f"Meilleure epoch ({best_epoch+1})")
ax2.set_title("Loss — Train vs Validation"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.legend(); ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(gs[1, 0])
gap = np.array(train_acc) - np.array(val_acc)
ax3.plot(epochs, gap, color="orange", linewidth=2)
ax3.axhline(0.05, color="red",  linestyle="--", label="Seuil 5 %")
ax3.axhline(0,    color="gray", linestyle=":")
ax3.fill_between(epochs, gap, 0, where=(gap >  0.05), color="red",   alpha=0.2, label="Zone overfitting")
ax3.fill_between(epochs, gap, 0, where=(gap <= 0.05), color="green", alpha=0.1, label="Zone OK")
ax3.set_title("Écart accuracy train − val"); ax3.set_xlabel("Epoch"); ax3.set_ylabel("Gap")
ax3.legend(); ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[1, 1])
labels_bar = ["Train\n(meilleure epoch)", "Validation\n(meilleure epoch)", "Test"]
values_bar = [final_train_acc, best_val_acc, test_acc]
bars = ax4.bar(labels_bar, values_bar, color=["steelblue","orange","green"], edgecolor="white", width=0.5)
ax4.set_ylim(min(values_bar) - 0.05, 1.0)
ax4.set_title("Accuracy finale : Train / Val / Test"); ax4.set_ylabel("Accuracy")
for bar, val in zip(bars, values_bar):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{val:.2%}", ha="center", fontweight="bold")
ax4.grid(axis="y", alpha=0.3)

plt.suptitle("Diagnostic Overfitting & Data Leakage", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Verdict ───────────────────────────────────────────────────
print("\n--------------------------VERDICT---------------------------\n")
issues = []
if gap_acc  > 0.05:  issues.append(f"OVERFITTING : écart accuracy train/val = {gap_acc:.1%} > 5 %")
if gap_loss > 0.1:   issues.append(f"OVERFITTING : écart loss train/val = {gap_loss:.3f} > 0.1")
if best_val_acc - test_acc > 0.05: issues.append("OPTIMISME : val accuracy trop supérieure au test accuracy")

if issues:
    print(" Problèmes détectés :")
    for issue in issues: print(f"  • {issue}")
else:
    print(" Aucun problème détecté.")
    print("  Le modèle généralise correctement et les données sont propres.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import tensorflow as tf
import itertools, os

print("=" * 65)
print("   DIAGNOSTIC COMPLET — FIRE DETECTION PROJECT")
print("=" * 65)

# ═══════════════════════════════════════════════════════════════
# 1. DATASET
# ═══════════════════════════════════════════════════════════════
print("\n▌ 1. DATASET\n")

total      = len(image_df)
n_fire     = (image_df.Label == "fire_images").sum()
n_nofire   = (image_df.Label == "non_fire_images").sum()
ratio      = n_fire / n_nofire

print(f"  Total images           : {total}")
print(f"  fire_images            : {n_fire}  ({n_fire/total*100:.1f} %)")
print(f"  non_fire_images        : {n_nofire}  ({n_nofire/total*100:.1f} %)")
print(f"  Ratio feu/non-feu      : {ratio:.2f}:1  {' déséquilibre' if ratio > 2 else ' acceptable'}")

# Extensions
exts = image_df.Filepath.apply(lambda p: os.path.splitext(p)[1].lower()).value_counts()
print(f"\n  Extensions             : {dict(exts)}")

# Tailles d'images (échantillon de 30)
sample_sizes = []
for p in image_df.Filepath.sample(30, random_state=42):
    try:
        w, h = Image.open(p).size
        sample_sizes.append((w, h))
    except Exception:
        pass
widths  = [s[0] for s in sample_sizes]
heights = [s[1] for s in sample_sizes]
print(f"  Largeur (30 échantil.) : min={min(widths)}  max={max(widths)}  moy={np.mean(widths):.0f}")
print(f"  Hauteur (30 échantil.) : min={min(heights)} max={max(heights)} moy={np.mean(heights):.0f}")

# ═══════════════════════════════════════════════════════════════
# 2. SPLIT & DATA LEAKAGE
# ═══════════════════════════════════════════════════════════════
print("\n▌ 2. SPLIT & DATA LEAKAGE\n")

train_paths = set(train_df.Filepath.astype(str))
test_paths  = set(test_df.Filepath.astype(str))
overlap     = train_paths & test_paths
union       = train_paths | test_paths

print(f"  train_df               : {len(train_df)} images  ({len(train_df)/total*100:.1f} %)")
print(f"  test_df                : {len(test_df)} images  ({len(test_df)/total*100:.1f} %)")
print(f"  Union = total          : {' OK' if len(union) == total else ' ANOMALIE'}")
print(f"  Chevauchement          : {len(overlap)} image(s)  {' Aucune fuite' if not overlap else ' FUITE DÉTECTÉE'}")

for split_name, split_df in [("train", train_df), ("test", test_df)]:
    c = split_df.Label.value_counts()
    print(f"  Répartition {split_name:5s}      : " +
          " | ".join(f"{k}: {v} ({v/len(split_df)*100:.0f}%)" for k, v in c.items()))

# ═══════════════════════════════════════════════════════════════
# 3. ARCHITECTURE DU MODÈLE
# ═══════════════════════════════════════════════════════════════
print("\n▌ 3. ARCHITECTURE DU MODÈLE\n")

total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
frozen_params    = total_params - trainable_params

print(f"  Backbone               : MobileNetV2 (ImageNet)")
print(f"  Paramètres totaux      : {total_params:,}")
print(f"  Paramètres entraînés   : {trainable_params:,}  ({trainable_params/total_params*100:.1f} %)")
print(f"  Paramètres gelés       : {frozen_params:,}  ({frozen_params/total_params*100:.1f} %)")
print(f"  Sortie                 : Dense(1, sigmoid) — classification binaire")
print(f"  Loss                   : binary_crossentropy")
print(f"  Optimizer              : Adam(lr=1e-4)")

# ═══════════════════════════════════════════════════════════════
# 4. ENTRAÎNEMENT & OVERFITTING
# ═══════════════════════════════════════════════════════════════
print("\n▌ 4. ENTRAÎNEMENT & OVERFITTING\n")

train_acc  = history.history["accuracy"]
val_acc    = history.history["val_accuracy"]
train_loss = history.history["loss"]
val_loss   = history.history["val_loss"]
n_epochs   = len(train_acc)
best_ep    = int(np.argmin(val_loss))

gap_acc  = train_acc[best_ep]  - val_acc[best_ep]
gap_loss = val_loss[best_ep]   - train_loss[best_ep]

print(f"  Epochs exécutées       : {n_epochs}  (EarlyStopping patience=5)")
print(f"  Meilleure epoch        : {best_ep + 1}")
print(f"  Train accuracy         : {train_acc[best_ep]:.4f}")
print(f"  Val   accuracy         : {val_acc[best_ep]:.4f}")
print(f"  Écart acc train/val    : {gap_acc:+.4f}   {' overfitting' if gap_acc > 0.05 else ' OK'}")
print(f"  Écart loss val/train   : {gap_loss:+.4f}   {' overfitting' if gap_loss > 0.10 else ' OK'}")

# Tendance sur les 3 dernières epochs
if n_epochs >= 3:
    trend_val = val_acc[-1] - val_acc[-3]
    print(f"  Tendance val_acc (3 dernières epochs) : {trend_val:+.4f}  "
          f"{'plateau/déclin' if trend_val <= 0 else 'amélioration'}")

# ═══════════════════════════════════════════════════════════════
# 5. ÉVALUATION SUR LE TEST SET
# ═══════════════════════════════════════════════════════════════
print("\n▌ 5. ÉVALUATION SUR LE TEST SET\n")

test_results = model.evaluate(test_images, verbose=0)
test_loss_val, test_acc_val = test_results[0], test_results[1]

# Prédictions complètes
pred_proba  = model.predict(test_images, verbose=0).flatten()
pred_binary = (pred_proba >= optimal_threshold).astype(int)
class_names = list(train_images.class_indices.keys())
idx_to_cls  = {v: k for k, v in train_images.class_indices.items()}
y_pred_str  = [idx_to_cls[p] for p in pred_binary]
y_true_str  = list(test_df.Label)

report_dict = classification_report(y_true_str, y_pred_str, output_dict=True)
cm = confusion_matrix(y_true_str, y_pred_str, labels=class_names)

print(f"  Test loss              : {test_loss_val:.4f}")
print(f"  Test accuracy          : {test_acc_val:.4f}  ({test_acc_val*100:.2f} %)")
print(f"  Écart val/test acc     : {val_acc[best_ep] - test_acc_val:+.4f}  "
      f"{' val trop optimiste' if val_acc[best_ep] - test_acc_val > 0.05 else ' OK'}")
print()
for cls in class_names:
    r = report_dict.get(cls, {})
    print(f"  [{cls}]")
    print(f"    Precision  : {r.get('precision', 0):.4f}")
    print(f"    Recall     : {r.get('recall', 0):.4f}")
    print(f"    F1-score   : {r.get('f1-score', 0):.4f}")
    print(f"    Support    : {int(r.get('support', 0))}")

print(f"\n  Macro F1               : {report_dict['macro avg']['f1-score']:.4f}")
print(f"  Weighted F1            : {report_dict['weighted avg']['f1-score']:.4f}")

tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
print(f"\n  Vrais  Positifs (feu correct)    : {tp}")
print(f"  Vrais  Négatifs (non-feu correct): {tn}")
print(f"  Faux   Positifs (fausse alarme)  : {fp}")
print(f"  Faux   Négatifs (feu manqué !)   : {fn}  {' CRITIQUE' if fn > 5 else 'OK'}")

# ═══════════════════════════════════════════════════════════════
# 6. ANALYSE DE CONFIANCE
# ═══════════════════════════════════════════════════════════════
print("\n▌ 6. CONFIANCE DES PRÉDICTIONS\n")

confidence = np.where(pred_proba >= optimal_threshold, pred_proba, 1 - pred_proba)
uncertain  = (confidence < 0.7).sum()
very_sure  = (confidence > 0.95).sum()

print(f"  Confiance moyenne      : {confidence.mean():.4f}")
print(f"  Confiance médiane      : {np.median(confidence):.4f}")
print(f"  Prédictions < 70 %     : {uncertain} / {len(confidence)}  (zone d'incertitude)")
print(f"  Prédictions > 95 %     : {very_sure} / {len(confidence)}  (très sûres)")

# Confiance sur les erreurs vs bonnes prédictions
correct_mask   = np.array(y_true_str) == np.array(y_pred_str)
conf_correct   = confidence[correct_mask].mean()
conf_incorrect = confidence[~correct_mask].mean() if (~correct_mask).any() else float("nan")
print(f"  Confiance moy (bonne pred)  : {conf_correct:.4f}")
print(f"  Confiance moy (erreur)      : {conf_incorrect:.4f}  "
      f"{'— modèle trop confiant sur ses erreurs ' if conf_incorrect > 0.7 else '— modèle hésitant sur ses erreurs '}")

# ═══════════════════════════════════════════════════════════════
# 7. GRAPHIQUES COMPLETS
# ═══════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 22))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)
epochs_x = range(1, n_epochs + 1)

# 7a. Accuracy
ax = fig.add_subplot(gs[0, 0])
ax.plot(epochs_x, train_acc, label="Train", lw=2)
ax.plot(epochs_x, val_acc,   label="Validation", lw=2, ls="--")
ax.axvline(best_ep + 1, color="gray", ls=":", lw=1.5, label=f"Best epoch {best_ep+1}")
ax.set_title("Accuracy Train / Val"); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)

# 7b. Loss
ax = fig.add_subplot(gs[0, 1])
ax.plot(epochs_x, train_loss, label="Train", lw=2)
ax.plot(epochs_x, val_loss,   label="Validation", lw=2, ls="--")
ax.axvline(best_ep + 1, color="gray", ls=":", lw=1.5)
ax.set_title("Loss Train / Val"); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)

# 7c. Gap accuracy (signal overfitting)
ax = fig.add_subplot(gs[0, 2])
gap_curve = np.array(train_acc) - np.array(val_acc)
ax.plot(epochs_x, gap_curve, color="orange", lw=2)
ax.axhline(0.05, color="red", ls="--", label="Seuil 5 %")
ax.axhline(0,    color="gray", ls=":")
ax.fill_between(epochs_x, gap_curve, 0, where=(gap_curve > 0.05), color="red",   alpha=0.25, label="Overfitting")
ax.fill_between(epochs_x, gap_curve, 0, where=(gap_curve <= 0.05), color="green", alpha=0.1,  label="OK")
ax.set_title("Gap accuracy train − val"); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)

# 7d. Barres Train / Val / Test
ax = fig.add_subplot(gs[1, 0])
vals  = [train_acc[best_ep], val_acc[best_ep], test_acc_val]
bars  = ax.bar(["Train", "Validation", "Test"], vals,
               color=["steelblue", "orange", "seagreen"], edgecolor="white", width=0.5)
ax.set_ylim(max(0, min(vals) - 0.08), 1.0)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005, f"{v:.2%}",
            ha="center", fontweight="bold", fontsize=10)
ax.set_title("Accuracy par split"); ax.set_ylabel("Accuracy"); ax.grid(axis="y", alpha=0.3)

# 7e. Matrice de confusion
ax = fig.add_subplot(gs[1, 1])
cax = ax.matshow(cm, cmap=plt.cm.Blues)
fig.colorbar(cax, ax=ax)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(class_names, rotation=25, ha="left", fontsize=8)
ax.set_yticklabels(class_names, fontsize=8)
ax.xaxis.set_label_position("bottom"); ax.xaxis.tick_bottom()
thresh = cm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, f"{cm[i,j]}", ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black", fontsize=14, fontweight="bold")
ax.set_title("Matrice de confusion"); ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")

# 7f. Distribution de confiance
ax = fig.add_subplot(gs[1, 2])
ax.hist(confidence[correct_mask],  bins=20, alpha=0.7, color="green",  label="Correct")
ax.hist(confidence[~correct_mask], bins=20, alpha=0.7, color="red",    label="Erreur")
ax.axvline(0.7,  color="orange", ls="--", label="Seuil 70 %")
ax.axvline(0.95, color="blue",   ls="--", label="Seuil 95 %")
ax.set_title("Distribution de confiance"); ax.set_xlabel("Confiance"); ax.legend(); ax.grid(alpha=0.3)

# 7g. Distribution des classes — dataset
ax = fig.add_subplot(gs[2, 0])
ax.bar(["fire_images", "non_fire_images"], [n_fire, n_nofire],
       color=["tomato", "steelblue"], edgecolor="white", width=0.5)
ax.set_title("Distribution classes — dataset complet"); ax.set_ylabel("Nb images"); ax.grid(axis="y", alpha=0.3)

# 7h. Précision / Rappel / F1 par classe
ax = fig.add_subplot(gs[2, 1])
metrics_labels = ["Precision", "Recall", "F1-score"]
x = np.arange(len(metrics_labels)); width = 0.35
for i, cls in enumerate(class_names):
    r = report_dict[cls]
    vals_cls = [r["precision"], r["recall"], r["f1-score"]]
    ax.bar(x + i * width, vals_cls, width, label=cls, alpha=0.85)
ax.set_xticks(x + width / 2); ax.set_xticklabels(metrics_labels)
ax.set_ylim(0, 1.1); ax.set_title("Métriques par classe")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

# 7i. Erreurs : faux positifs et faux négatifs
ax = fig.add_subplot(gs[2, 2])
error_mask = ~correct_mask
n_errors   = error_mask.sum()
err_labels = np.array(y_true_str)[error_mask]
err_counts = pd.Series(err_labels).value_counts()
if n_errors > 0:
    ax.bar(err_counts.index, err_counts.values, color=["tomato", "steelblue"][:len(err_counts)], edgecolor="white")
    ax.set_title(f"Erreurs par classe vraie ({n_errors} total)")
    ax.set_ylabel("Nb erreurs"); ax.grid(axis="y", alpha=0.3)
else:
    ax.text(0.5, 0.5, "0 erreur !", ha="center", va="center", fontsize=20, color="green")
    ax.set_title("Erreurs"); ax.axis("off")

# 7j. Exemples d'erreurs (jusqu'à 6)
error_indices = np.where(error_mask)[0]
n_show = min(len(error_indices), 6)
if n_show > 0:
    ax_err = [fig.add_subplot(gs[3, col]) for col in range(3)]
    shown = 0
    for idx in error_indices[:n_show]:
        if shown >= 3:
            break
        path = test_df.Filepath.iloc[idx]
        try:
            img = np.array(Image.open(path).resize((224, 224)).convert("RGB"))
            ax_err[shown].imshow(img / 255.0)
            ax_err[shown].set_title(
                f"Vrai : {y_true_str[idx].replace('_images','')}\n"
                f"Prédit : {y_pred_str[idx].replace('_images','')} ({confidence[idx]:.0%})",
                color="red", fontsize=9)
            ax_err[shown].axis("off")
            shown += 1
        except Exception:
            pass
    for remaining in range(shown, 3):
        ax_err[remaining].axis("off")
    fig.text(0.5, 0.01, "Exemples d'erreurs du modèle", ha="center", fontsize=12, color="red")
else:
    for col in range(3):
        ax_z = fig.add_subplot(gs[3, col])
        ax_z.text(0.5, 0.5, "Aucune erreur", ha="center", va="center", fontsize=14, color="green")
        ax_z.axis("off")

plt.suptitle("DIAGNOSTIC COMPLET — Fire Detection Model", fontsize=16, fontweight="bold", y=1.005)
plt.savefig("diagnostic_complet.png", dpi=120, bbox_inches="tight")
plt.show()

# ═══════════════════════════════════════════════════════════════
# 8. VERDICT FINAL
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("   VERDICT FINAL")
print("=" * 65 + "\n")

issues = []
goods  = []

# Données
if not overlap:         goods.append("Aucune fuite de données (train ∩ test = ∅)")
else:                   issues.append(f"DATA LEAKAGE : {len(overlap)} image(s) commune(s)")
if ratio <= 3:          goods.append(f"Déséquilibre modéré ({ratio:.1f}:1) — class_weight appliqué")
else:                   issues.append(f"Déséquilibre fort ({ratio:.1f}:1) — considérer oversampling")

# Modèle
if gap_acc <= 0.05:     goods.append(f"Pas d'overfitting (écart acc={gap_acc:.1%})")
else:                   issues.append(f"Overfitting : écart accuracy={gap_acc:.1%} > 5 %")
if gap_loss <= 0.10:    goods.append(f"Loss stable (écart loss={gap_loss:.3f})")
else:                   issues.append(f"Overfitting : écart loss={gap_loss:.3f} > 0.10")

# Performance
if test_acc_val >= 0.95: goods.append(f"Excellente accuracy test ({test_acc_val:.2%})")
elif test_acc_val >= 0.90: goods.append(f"Bonne accuracy test ({test_acc_val:.2%})")
else:                    issues.append(f"Accuracy test faible ({test_acc_val:.2%})")

wf1 = report_dict["weighted avg"]["f1-score"]
if wf1 >= 0.95:         goods.append(f"Excellent F1 pondéré ({wf1:.4f})")
elif wf1 >= 0.90:       goods.append(f"Bon F1 pondéré ({wf1:.4f})")
else:                   issues.append(f"F1 pondéré insuffisant ({wf1:.4f})")

if fn <= 3:             goods.append(f"Peu de feux manqués — Faux Négatifs = {fn}")
elif fn <= 8:           issues.append(f"Attention : {fn} feux non détectés (FN)")
else:                   issues.append(f"CRITIQUE : {fn} feux manqués (FN élevé)")

if conf_incorrect > 0.75:
    issues.append(f"Modèle trop confiant sur ses erreurs (conf moy = {conf_incorrect:.0%})")
else:
    goods.append(f"Confiance calibrée sur les erreurs ({conf_incorrect:.0%})")

print("  Points positifs :")
for g in goods:  print(f"      • {g}")
print(f"\n  {' Points à corriger :' if issues else '  Aucun problème détecté !'}")
for i in issues: print(f"      • {i}")
print(f"\n  Graphique sauvegardé : diagnostic_complet.png")
print("=" * 65)

In [ ]:
# Analyse détaillée des erreurs les plus confiantes
error_indices = np.where(np.array(y_true_str) != np.array(y_pred_str))[0]
error_conf    = confidence[error_indices]

# Trier par confiance décroissante
sorted_idx = error_indices[np.argsort(error_conf)[::-1]]

print(f"Total erreurs : {len(error_indices)}\n")
print(f"{'#':<4} {'Confiance':>10} {'Vrai label':>20} {'Prédit':>20}")
print("-" * 58)
for rank, idx in enumerate(sorted_idx):
    print(f"{rank+1:<4} {confidence[idx]:>10.1%} "
          f"{y_true_str[idx]:>20} {y_pred_str[idx]:>20}")

# Afficher les images les plus mal classées avec le plus de confiance
n_show = min(len(sorted_idx), 6)
cols   = 3
rows   = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 5))
axes = axes.flat if n_show > 1 else [axes]

for i, idx in enumerate(sorted_idx[:n_show]):
    img = np.array(Image.open(test_df.Filepath.iloc[idx]).resize((224, 224)).convert("RGB"))
    axes[i].imshow(img / 255.0)
    axes[i].set_title(
        f"Vrai  : {y_true_str[idx].replace('_images', '')}\n"
        f"Prédit : {y_pred_str[idx].replace('_images', '')}\n"
        f"Confiance : {confidence[idx]:.1%}",
        color="red", fontsize=11, fontweight="bold"
    )
    axes[i].axis("off")

for j in range(n_show, rows * cols):
    axes[j].axis("off")

plt.suptitle("Erreurs les plus confiantes — cas difficiles pour le modèle",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# AUDIT DES LABELS — Détection semi-automatique d'annotations suspectes
# Principe : une image très confiante dans la mauvaise direction est
# un candidat sérieux pour être mal étiquetée dans le dataset.
# ═══════════════════════════════════════════════════════════════
print("=" * 62)
print("   AUDIT DES LABELS — Annotations potentiellement erronées")
print("=" * 62)

AUDIT_THRESHOLD = 0.80  # seuil de confiance pour signaler une image suspecte

# Créer un générateur sur l'ensemble du dataset (train + test, sans shuffle)
full_generator = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)
full_images_audit = full_generator.flow_from_dataframe(
    dataframe=image_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMAGE_SIZE,
    color_mode='rgb',
    class_mode='binary',
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Prédictions sur le dataset complet
print("\nPrédictions en cours sur tout le dataset...")
all_proba = model.predict(full_images_audit, verbose=1).flatten()
all_labels_audit = list(image_df.Label)
all_paths_audit  = list(image_df.Filepath)

fire_class_idx_audit = full_images_audit.class_indices['fire_images']

# Score = probabilité d'être feu (indépendant de l'ordre alphabétique des classes)
all_fire_score = all_proba if fire_class_idx_audit == 1 else 1 - all_proba

# Identifier les suspects
suspects_fire_in_nonfire = []  # étiquette non_fire, mais modèle très sûr que c'est du feu
suspects_nonfire_in_fire = []  # étiquette fire, mais modèle très sûr que ce n'est pas du feu

for path, label, fire_score in zip(all_paths_audit, all_labels_audit, all_fire_score):
    if label == 'non_fire_images' and fire_score >= AUDIT_THRESHOLD:
        suspects_fire_in_nonfire.append((path, fire_score))
    elif label == 'fire_images' and (1 - fire_score) >= AUDIT_THRESHOLD:
        suspects_nonfire_in_fire.append((path, 1 - fire_score))

# Rapport texte
print(f"\n{'─'*62}")
print(f"Images dans non_fire_images prédites FEU (≥{AUDIT_THRESHOLD:.0%} confiance) : "
      f"{len(suspects_fire_in_nonfire)}")
for p, s in sorted(suspects_fire_in_nonfire, key=lambda x: -x[1]):
    print(f"  {s:.1%}  {p}")

print(f"\nImages dans fire_images prédites NON-FEU (≥{AUDIT_THRESHOLD:.0%} confiance) : "
      f"{len(suspects_nonfire_in_fire)}")
for p, s in sorted(suspects_nonfire_in_fire, key=lambda x: -x[1]):
    print(f"  {s:.1%}  {p}")
print(f"{'─'*62}")

# Affichage visuel des suspects (jusqu'à 9)
all_suspects_audit = (
    [(p, "non_fire → FEU ?",   s, "tomato")     for p, s in suspects_fire_in_nonfire] +
    [(p, "fire → NON-FEU ?",   s, "steelblue")  for p, s in suspects_nonfire_in_fire]
)
all_suspects_audit.sort(key=lambda x: -x[2])

if all_suspects_audit:
    n_show = min(len(all_suspects_audit), 9)
    cols = 3
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 5))
    axes_flat = list(axes.flat) if hasattr(axes, 'flat') else [axes]

    for i, (path, title, score, color) in enumerate(all_suspects_audit[:n_show]):
        try:
            img = np.array(Image.open(path).resize((224, 224)).convert("RGB"))
            axes_flat[i].imshow(img / 255.0)
            axes_flat[i].set_title(
                f"{title}\nConfiance modèle : {score:.1%}\n{Path(path).name}",
                color=color, fontsize=9, fontweight="bold"
            )
        except Exception as e:
            axes_flat[i].text(0.5, 0.5, f"Erreur:\n{e}", ha="center", va="center")
        axes_flat[i].axis("off")

    for j in range(n_show, rows * cols):
        axes_flat[j].axis("off")

    total_suspects = len(all_suspects_audit)
    plt.suptitle(
        f"Audit labels — {total_suspects} image(s) suspecte(s) sur {len(image_df)} "
        f"(seuil confiance ≥ {AUDIT_THRESHOLD:.0%})",
        fontsize=12, fontweight="bold", color="darkred"
    )
    plt.tight_layout()
    plt.show()

    print(f"\nAction recommandée : inspecter manuellement les {total_suspects} image(s) ci-dessus")
    print("et corriger leur dossier si le label est incorrect.")
else:
    print(f"\nAucune image suspecte détectée avec un seuil de {AUDIT_THRESHOLD:.0%}.")
    print("Les labels du dataset semblent cohérents avec les prédictions du modèle.")